# EDA, Model Comparisons, and GradCAM

Exploratory analysis for the solar panel fault classifier. Reusable logic
(`get_dataloaders`, `build_model`, `train_model`, `evaluate_model`) is imported from `src/`.

The main model in `scripts/train.py` uses stratified sampling. This notebook compares it
against two alternatives: an untrained ResNet18 baseline, and the same model trained on a
simple random split.

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import os

from src.config import load_config
from src.data import get_dataloaders
from src.model import build_model
from src.train import get_device, train_model
from src.evaluate import evaluate_model, plot_confusion_matrix

cfg = load_config()

device = get_device()
train_loader, test_loader, dataset = get_dataloaders(cfg)
class_names = dataset.classes


## Data Visualization

Class distribution and sample images from each category.

In [ ]:
# Bar graph for image counts by class
DATA_DIR = cfg['data']['data_dir']
class_counts = {}

for class_name in sorted(os.listdir(DATA_DIR)):
    class_path = os.path.join(DATA_DIR, class_name)
    if os.path.isdir(class_path):
        image_count = len([
            f for f in os.listdir(class_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])
        class_counts[class_name] = image_count

print(class_counts)

plt.figure(figsize=(10, 6))
plt.bar(class_counts.keys(), class_counts.values())
plt.title("Number of Images per Solar Panel Category")
plt.xlabel("Solar Panel Category")
plt.ylabel("Number of Images")
plt.xticks(rotation=30)
for i, count in enumerate(class_counts.values()):
    plt.text(i, count + 2, str(count), ha="center")
plt.tight_layout()
plt.show()


In [ ]:
# Sample images from each class
from PIL import Image

classes = sorted([
    folder for folder in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, folder))
])

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, class_name in zip(axes.flatten(), classes):
    class_path = os.path.join(DATA_DIR, class_name)
    image_file = next(
        f for f in os.listdir(class_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    )
    image_path = os.path.join(class_path, image_file)
    img = Image.open(image_path)
    ax.imshow(img)
    display_name = class_name.replace("-", " ").replace("_", " ")
    ax.set_title(display_name, fontsize=12)
    ax.axis("off")

plt.suptitle("Sample Images from Each Solar Panel Category", fontsize=16)
plt.tight_layout()
plt.show()


## Compare to Baseline Model

The baseline model is ResNet18 with no additional training, to estimate the efficacy of fine-tuning.

In [ ]:
criterion = nn.CrossEntropyLoss()

baseline_model = build_model(cfg['model']['num_classes']).to(device)

baseline_test_loss, baseline_test_accuracy, baseline_class_accuracy, baseline_all_labels, baseline_all_preds = evaluate_model(
    baseline_model, criterion, test_loader, device, class_names
)

plot_confusion_matrix(baseline_all_labels, baseline_all_preds, class_names)


## Compare to Simple Random Sampling Model

The main model (`scripts/train.py`) uses **stratified sampling**, which keeps each class's
proportion consistent across train and test. This section trains the same architecture on a
**simple random split** instead, to show what stratification buys us.

Because the class counts are uneven, a random split can leave some classes under-represented
in the test set, making per-class accuracy noisier and harder to trust.

In [ ]:
from src.data import get_random_split_dataloaders

random_train_loader, random_test_loader, _ = get_random_split_dataloaders(cfg)

random_model = build_model(cfg['model']['num_classes']).to(device)
random_optimizer = optim.Adam(random_model.parameters(), lr=cfg['train']['learning_rate'])

random_epoch_accuracies = train_model(
    random_model, criterion, random_optimizer, random_train_loader, device, epochs=cfg['train']['epochs']
)

random_test_loss, random_test_accuracy, random_class_accuracy, random_all_labels, random_all_preds = evaluate_model(
    random_model, criterion, random_test_loader, device, class_names
)

plot_confusion_matrix(random_all_labels, random_all_preds, class_names)

## GradCAM (XAI)

Requires a trained `model` in scope — run `scripts/train.py` first, or train inline here, then load its weights.

In [ ]:
from src.explain import apply_gradcam

model = build_model(cfg['model']['num_classes']).to(device)
model.load_state_dict(torch.load(cfg['output']['model_path'], map_location=device))

target_layer = model.layer4[-1]

for i in range(6):
    image, label = test_loader.dataset[i]
    image_tensor = image.unsqueeze(0).to(device)
    print(f"Image {i} — True label: {class_names[label]}")
    apply_gradcam(model, image_tensor, target_layer, device)
